In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ── Load ──────────────────────────────────────────────────────
train_df = pd.read_csv('../../data/splits/train.csv', low_memory=False)
test_df  = pd.read_csv('../../data/splits/test.csv',  low_memory=False)

TARGET = 'Price_log'

DROP_COLS  = [
    'Price_log', 'Price_original', 'demand_label',
    'demand_label_3', 'demand_score',
]
DROP_COLS += [c for c in train_df.columns if c.endswith('_raw')]

feature_cols = [c for c in train_df.columns if c not in DROP_COLS]

# Drop high-VIF for linear models
linear_features = [c for c in feature_cols if c != 'Accommodates']

X_train = train_df[feature_cols]
y_train = train_df[TARGET]
X_test  = test_df[feature_cols]
y_test  = test_df[TARGET]

X_train_lin = train_df[linear_features]
X_test_lin  = test_df[linear_features]

# ── Evaluate helper ───────────────────────────────────────────
def evaluate(name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    # Convert RMSE back to dollars
    rmse_dollars = np.expm1(rmse)
    print(f"\n=== {name} ===")
    print(f"  RMSE        : {rmse:.4f} (log scale)")
    print(f"  RMSE        : ${rmse_dollars:.2f} (dollar scale)")
    print(f"  MAE         : {mae:.4f}")
    print(f"  R²          : {r2:.4f}")

# ── Model 1: Linear Regression ────────────────────────────────
lr = LinearRegression()
lr.fit(X_train_lin, y_train)
evaluate("Linear Regression", y_test, lr.predict(X_test_lin))

# ── Model 2: Ridge Regression ─────────────────────────────────
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_lin, y_train)
evaluate("Ridge Regression", y_test, ridge.predict(X_test_lin))

# ── Model 3: Random Forest Regressor ─────────────────────────
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
evaluate("Random Forest", y_test, rf.predict(X_test))

# ── Feature importance (answers Q1 directly) ─────────────────
importance = pd.Series(rf.feature_importances_, index=feature_cols)
print("\nTop 15 features by importance (answers Q1):")
print(importance.sort_values(ascending=False).head(15))


=== Linear Regression ===
  RMSE        : 0.5039 (log scale)
  RMSE        : $0.66 (dollar scale)
  MAE         : 0.3736
  R²          : 0.5909

=== Ridge Regression ===
  RMSE        : 0.5039 (log scale)
  RMSE        : $0.66 (dollar scale)
  MAE         : 0.3736
  R²          : 0.5909

=== Random Forest ===
  RMSE        : 0.3245 (log scale)
  RMSE        : $0.38 (dollar scale)
  MAE         : 0.2352
  R²          : 0.8304

Top 15 features by importance (answers Q1):
Cleaning Fee                 0.276928
Room Type_Entire home/apt    0.127662
Latitude                     0.087822
Country Code                 0.068303
Longitude                    0.063695
Bathrooms                    0.038668
Zipcode                      0.031131
Country                      0.030636
Weekly Price                 0.026117
Accommodates                 0.024124
Bedrooms                     0.016345
Monthly Price                0.010714
Extra People                 0.010437
Neighbourhood Cleansed       0.